<p align="center">
  <img src="https://i0.wp.com/www.tiempodecine.co/web/wp-content/uploads/2015/11/Robert-De-Niro-in-Taxi-Driver-1976.jpg?resize=750%2C375&ssl=1" style="width:100%; max-width:900px; height:180px; object-fit:cover; border-radius:10px;"/>
</p>
<div style="text-align:center;">
  <h1 style="color:#FFD700; display:inline-block; margin:0;">Optimización del Transporte en Nueva York</h1>
  <p>
    <b>Green Taxi | Machine Learning & Data Science | CRISP-DM</b><br>
    <span style="font-size:1.1em;">Fase 6: Deployment (Despliegue)</span>
  </p>
</div>

# FASE 6: Deployment (Despliegue)

## Objetivo
El objetivo de esta fase es poner en producción el modelo de clustering seleccionado (K-Means) para que pueda ser consumido por otros sistemas o usuarios. Se ha desarrollado una API REST utilizando **FastAPI** y se ha contenedorizado la aplicación usando **Docker** y **Docker Compose**.

## Arquitectura de Despliegue

La solución consta de los siguientes componentes:
1.  **API REST (FastAPI)**: Expone el modelo de Machine Learning para realizar predicciones (asignación de clusters).
2.  **Base de Datos (PostgreSQL)**: Incluida en la arquitectura para futuro almacenamiento de logs o metadatos (opcional en esta etapa).
3.  **Docker**: Para empaquetar la aplicación y sus dependencias.

### Estructura del Proyecto
```
deployment/
├── app/
│   ├── main.py              # Código de la aplicación FastAPI
│   └── models_assets/       # Modelos entrenados y scalers
│       ├── kmeans_model.pkl
│       └── feature_scaler.pkl
├── Dockerfile               # Definición de la imagen Docker
├── docker-compose.yml       # Orquestación de servicios (API + DB)
└── requirements.txt         # Dependencias de Python
```

## Código de la Aplicación (FastAPI)

A continuación se muestra el código principal de la API (`deployment/app/main.py`).

In [1]:
from IPython.display import Code
Code(filename='deployment/app/main.py', language='python')

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import joblib
import pandas as pd
import numpy as np
import os

app = FastAPI(title="NYC Taxi Zone Clustering API", description="API for predicting cluster assignment of NYC Taxi Zones", version="1.0.0")

# Load model and scaler
MODEL_PATH = "app/models_assets/kmeans_model.pkl"
SCALER_PATH = "app/models_assets/feature_scaler.pkl"

model = None
scaler = None

@app.on_event("startup")
def load_assets():
    global model, scaler
    try:
        model = joblib.load(MODEL_PATH)
        scaler = joblib.load(SCALER_PATH)
        print("Model and scaler loaded successfully.")
    except Exception as e:
        print(f"Error loading model or scaler: {e}")
        # In production, you might want to raise an error or exit
        pass

class ZoneFeatures(BaseModel):
    total_trips: float
    avg_pickup_hour: float
    std_pickup_hour: float
    peak_hour: float
    peak_hour_concentration_pct: float
    pct_AM_Peak: float
    pct_PM_Peak: float
    pct_OP_day: float
    pct_OP_night: float
    pct_Early_Morning: float
    pct_weekend_trips: float
    avg_trip_distance: float
    median_trip_distance: float
    avg_trip_duration: float
    avg_fare: float
    avg_total_amount: float
    avg_tip: float
    avg_tolls: float
    pct_yellow_taxi: float

@app.get("/")
def read_root():
    return {"message": "Welcome to the NYC Taxi Zone Clustering API. Use /predict to get cluster assignments."}

@app.post("/predict")
def predict_cluster(features: ZoneFeatures):
    if model is None or scaler is None:
        raise HTTPException(status_code=500, detail="Model or scaler not loaded.")
    
    try:
        # Convert features to dataframe/array in the correct order
        data = {
            "total_trips": [features.total_trips],
            "avg_pickup_hour": [features.avg_pickup_hour],
            "std_pickup_hour": [features.std_pickup_hour],
            "peak_hour": [features.peak_hour],
            "peak_hour_concentration_pct": [features.peak_hour_concentration_pct],
            "pct_AM_Peak": [features.pct_AM_Peak],
            "pct_PM_Peak": [features.pct_PM_Peak],
            "pct_OP_day": [features.pct_OP_day],
            "pct_OP_night": [features.pct_OP_night],
            "pct_Early_Morning": [features.pct_Early_Morning],
            "pct_weekend_trips": [features.pct_weekend_trips],
            "avg_trip_distance": [features.avg_trip_distance],
            "median_trip_distance": [features.median_trip_distance],
            "avg_trip_duration": [features.avg_trip_duration],
            "avg_fare": [features.avg_fare],
            "avg_total_amount": [features.avg_total_amount],
            "avg_tip": [features.avg_tip],
            "avg_tolls": [features.avg_tolls],
            "pct_yellow_taxi": [features.pct_yellow_taxi]
        }
        
        df = pd.DataFrame(data)
        
        # Scale features
        scaled_features = scaler.transform(df)
        
        # Predict
        cluster = model.predict(scaled_features)[0]
        
        return {
            "cluster": int(cluster),
            "features_received": features.dict()
        }
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

## Configuración de Docker

### Dockerfile
Define el entorno de ejecución de la API.

In [2]:
Code(filename='deployment/Dockerfile', language='docker')

FROM python:3.9-slim

WORKDIR /code

COPY ./requirements.txt /code/requirements.txt

RUN pip install --no-cache-dir --upgrade -r /code/requirements.txt

COPY ./app /code/app

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]

### Docker Compose
Define los servicios y la red.

In [3]:
Code(filename='deployment/docker-compose.yml', language='yaml')

version: '3.8'

services:
  api:
    build: .
    ports:
      - "8000:8000"
    volumes:
      - ./app:/code/app
    depends_on:
      - db
    environment:
      - DATABASE_URL=postgresql://user:password@db:5432/taxi_db

  db:
    image: postgres:15
    volumes:
      - postgres_data:/var/lib/postgresql/data
    environment:
      - POSTGRES_USER=user
      - POSTGRES_PASSWORD=password
      - POSTGRES_DB=taxi_db
    ports:
      - "5432:5432"

volumes:
  postgres_data:

## Instrucciones de Ejecución

Para levantar el servicio, ejecutar los siguientes comandos en la terminal desde la carpeta `deployment/`:

```bash
cd deployment
docker-compose up --build
```

La API estará disponible en: `http://localhost:8000`
Documentación interactiva (Swagger UI): `http://localhost:8000/docs`

## Ejemplo de Uso (Cliente Python)

Una vez que el servicio esté corriendo, puedes usar el siguiente script para probarlo:

In [5]:
%pip install requests
import requests
import json

# URL de la API
url = "http://localhost:8000/predict"

# Datos de ejemplo (una zona típica)
payload = {
    "total_trips": 150,
    "avg_pickup_hour": 14.5,
    "std_pickup_hour": 3.2,
    "peak_hour": 18.0,
    "peak_hour_concentration_pct": 0.15,
    "pct_AM_Peak": 0.2,
    "pct_PM_Peak": 0.3,
    "pct_OP_day": 0.4,
    "pct_OP_night": 0.1,
    "pct_Early_Morning": 0.0,
    "pct_weekend_trips": 0.25,
    "avg_trip_distance": 3.5,
    "median_trip_distance": 2.8,
    "avg_trip_duration": 15.5,
    "avg_fare": 12.5,
    "avg_total_amount": 15.0,
    "avg_tip": 2.0,
    "avg_tolls": 0.0,
    "pct_yellow_taxi": 0.1
}

try:
    response = requests.post(url, json=payload)
    if response.status_code == 200:
        print("Predicción exitosa!")
        print(json.dumps(response.json(), indent=2))
    else:
        print(f"Error: {response.status_code}")
        print(response.text)
except requests.exceptions.ConnectionError:
    print("No se pudo conectar a la API. Asegúrate de que Docker esté corriendo.")

  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
Using cached idna-3.11-py3-none-any.whl (71 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


No se pudo conectar a la API. Asegúrate de que Docker esté corriendo.
